# Week 9: Transformers & LLM Architectures
## Notebook 07: Multi-Query Attention (MQA) & FlashDecoding

### 1. Conceptual Intuition & Theoretical Background
During autoregressive generation in LLMs, each step generates a single new token ($Q \in \mathbb{R}^{B \times H \times 1 \times d}$). However, calculating attention requires loading the entire past Key-Value cache ($K, V \in \mathbb{R}^{B \times H \times S \times d}$) from high-bandwidth GPU memory (HBM) into SRAM.

At batch decoding time, the **arithmetic intensity** is very low:
$$\text{Operational Intensity} = \frac{\text{FLOPs}}{\text{Bytes Transferred}} \ll 1$$
Inference is completely **memory-bandwidth bound**.

**Multi-Query Attention (MQA)** shares a **single key and value head** across all query attention heads ($H_K = H_V = 1$):
$$\text{KV Cache Memory Reduction} = \frac{1}{H_Q} \times \text{MHA Memory}$$

**FlashDecoding** further parallelizes decoding by partitioning the long sequence length dimension $S$ across multiple GPU thread blocks and computing partial softmax reductions.

---

### 2. Scratch Python Implementation: MHA vs MQA Engine

In [ ]:
import numpy as np

def multi_head_attention(Q, K, V):
    """
    Standard MHA: Q, K, V all have H distinct heads.
    Shape: (Batch, Heads, SeqLen, HeadDim)
    """
    d_k = Q.shape[-1]
    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
    weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights /= np.sum(weights, axis=-1, keepdims=True)
    return np.matmul(weights, V)

def multi_query_attention(Q, K_single, V_single):
    """
    MQA: Q has H heads, but K and V have only 1 shared head.
    Q:        (Batch, Heads, SeqLen_Q, HeadDim)
    K_single: (Batch, 1, SeqLen_KV, HeadDim)
    V_single: (Batch, 1, SeqLen_KV, HeadDim)
    """
    d_k = Q.shape[-1]
    # Broadcast single KV head across all H Query heads
    scores = np.matmul(Q, K_single.transpose(0, 1, 3, 2)) / np.sqrt(d_k)
    weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    weights /= np.sum(weights, axis=-1, keepdims=True)
    return np.matmul(weights, V_single)

# Memory & Footprint Analysis
batch_size = 8
heads = 32
seq_len = 4096
head_dim = 128
dtype_bytes = 2 # FP16

mha_kv_bytes = 2 * (batch_size * heads * seq_len * head_dim * dtype_bytes)
mqa_kv_bytes = 2 * (batch_size * 1 * seq_len * head_dim * dtype_bytes)

print(f"MHA KV Cache Footprint (4K Context): {mha_kv_bytes / (1024 * 1024):.2f} MB")
print(f"MQA KV Cache Footprint (4K Context): {mqa_kv_bytes / (1024 * 1024):.2f} MB")
print(f"Memory Bandwidth Saved:              {(1 - mqa_kv_bytes / mha_kv_bytes) * 100:.2f}%")

### 3. Step-by-Step Code Walkthrough
1. **Shared Key/Value Projection**: All 32 query heads interact with the same underlying Key/Value representations.
2. **Broadcasting Semantics**: `matmul` broadcasts dimension 1 across heads without duplicating physical memory.
3. **Footprint Reduction**: Cuts memory traffic by $32\times$, directly increasing generation tokens-per-second throughput.

---

### 4. Industry Applications & Trade-offs
- **Ultra-Long Context Inference (e.g. Claude 3, PaLM):** MQA allows serving 100K+ token context windows within GPU VRAM limits.
- **Trade-off:** Slight degradation in expressive capacity compared to standard Multi-Head Attention.

---

### 5. Top Interview Questions & Answers
**Q1: Why is LLM generation during the decoding phase memory-bandwidth bound rather than compute bound?**
> **Answer:** In generation, each forward step takes a single new token ($Q=1$) and multiplies it against the entire KV cache of historical tokens. The ratio of math operations to bytes loaded is extremely low, meaning GPU cores sit idle waiting for memory retrieval.

**Q2: How does FlashDecoding differ from FlashAttention-2?**
> **Answer:** FlashAttention-2 parallelizes over batch and query heads, which is optimal for prefill ($Q \gg 1$). In decode ($Q=1$), this leaves GPU SMs underutilized. FlashDecoding adds a third parallel dimension across the KV sequence length with a final split-reduction.